# Options from Zero · Episode 13: Itô's lemma, without the pain

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The puzzle · 4. Taylor meets randomness · 5. Itô's lemma for the log price · 6. Mean versus median · 7. It's in d1 and d2 · 8. This episode's question · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
mu = 0.08   # the share's expected return (real world), for the simulation

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 13,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

## 2. Last episode's question

The share falls \$10. The line alone says the call loses \$5.14; it really loses only \$4.13. So the line-only estimate of the change is too low (too negative): the bend always adds back a positive $\tfrac12\,\Gamma\,(\Delta S)^2$.

In [ ]:
g = bs_greeks(S, K, T, r, vol)
exact = black_scholes(S - 10, K, T, r, vol) - g["price"]
out["answer"] = {"line": g["delta"] * 10, "exact": abs(exact), "move": 10}
out["answer"]

## 3. The puzzle

A share rises 50%, then falls 50%. The average of the two returns is 0%, but \$100 becomes \$150 and then \$75. Ups and downs of the same size don't cancel: in log returns (Episode 2), $\ln 1.5 + \ln 0.5 = \ln 0.75 < 0$.

In [ ]:
out["puzzle"] = {"start": 100.0, "after_up": 150.0, "after_down": 75.0, "left_share": 1.5 * 0.5, "loss_share": 1 - 1.5 * 0.5,
                 "up_pct": 50, "down_pct": 50, "avg_pct": (50 - 50) / 2, "log_sum": math.log(1.5) + math.log(0.5)}
out["puzzle"]

## 4. Taylor meets randomness

For any smooth $f(S)$, Taylor (Episode 12) gives $df \approx f'(S)\,dS + \tfrac12 f''(S)\,(dS)^2$. In ordinary calculus $(dS)^2$ is negligible. Here it isn't: with $dS = \mu S\,dt + \sigma S\,dW$ and $(dW)^2 = dt$ (Episode 11), $(dS)^2 \approx \sigma^2 S^2\,dt$, which adds up over a year to something the same size as the drift.

A year of daily moves in our share: the squared returns add up to about $\sigma^2 = 0.04$, while the squared drift parts add up to almost nothing.

In [ ]:
rng = np.random.default_rng(13)
steps, n = 252, 20_000
dt = 1 / steps
z = rng.standard_normal((n, steps))
rets = mu * dt + vol * math.sqrt(dt) * z                 # daily returns dS/S
sum_sq = (rets ** 2).sum(axis=1)
out["randomness"] = {"sum_sq_mean": float(sum_sq.mean()), "vol_sq": vol ** 2, "drift_sq": steps * (mu * dt) ** 2,
                     "sum_sq_sd": float(sum_sq.std())}
assert abs(sum_sq.mean() / vol ** 2 - 1) < 0.01
out["randomness"]

## 5. Itô's lemma for the log price

Take $f(S) = \ln S$: $f' = 1/S$ and $f'' = -1/S^2$. Then

$$d\ln S = \frac{dS}{S} - \frac{1}{2}\frac{(dS)^2}{S^2} = \Big(\mu - \frac{\sigma^2}{2}\Big)dt + \sigma\,dW.$$

The log price grows at $\mu - \sigma^2/2$, not $\mu$: **volatility drag**. A check that doesn't assume the answer: step the share price itself forward with daily returns $\mu\,dt + \sigma\sqrt{dt}\,Z$ (no logs anywhere), then measure the average log return.

In [ ]:
log_growth = np.log(np.prod(1 + rets, axis=1))           # ln(S_T / S_0), from simple daily steps
out["ito"] = {"mu": mu, "vol": vol, "theory": mu - vol ** 2 / 2, "simulated": float(log_growth.mean()),
              "se": float(log_growth.std() / math.sqrt(n)), "mu_pct": mu * 100, "drag_pct": vol ** 2 / 2 * 100,
              "theory_pct": (mu - vol ** 2 / 2) * 100}
assert abs(out["ito"]["simulated"] - out["ito"]["theory"]) < 4 * out["ito"]["se"] + 0.002
out["ito"]

## 6. Mean versus median

A million one-year outcomes with $\mu = 8\%$, $\sigma = 20\%$. The **mean** grows at the full $\mu$: $100e^{0.08} = 108.33$. The **median**, the typical outcome, grows at $\mu - \sigma^2/2 = 6\%$: $100e^{0.06} = 106.18$. A few big winners lift the mean above the typical outcome, and a large share of outcomes finish below the start.

In [ ]:
ST = S * np.exp((mu - vol ** 2 / 2) * T + vol * math.sqrt(T) * rng.standard_normal(1_000_000))
out["outcomes"] = {"mean": float(ST.mean()), "mean_theory": S * math.exp(mu * T), "median": float(np.median(ST)),
                   "median_theory": S * math.exp((mu - vol ** 2 / 2) * T), "below_start": float((ST < S).mean()),
                   "below_theory": norm_cdf(-(mu - vol ** 2 / 2) * math.sqrt(T) / vol)}
ob = np.arange(40.0, 220.01, 4.0)
oc, _ = np.histogram(np.clip(ST, 40.0, 219.99), bins=ob)
out["outcomes"].update({"hist_x": ((ob[:-1] + ob[1:]) / 2).tolist(), "hist_share": (oc / len(ST)).round(6).tolist(), "bin": 4.0,
                        "start": S, "lo": 40.0,
                        "ticks": [{"x": float(x), "label": f"${x}"} for x in (40, 70, 100, 130, 160, 190, 220)]})
assert abs(out["outcomes"]["mean"] / out["outcomes"]["mean_theory"] - 1) < 0.001
assert abs(out["outcomes"]["median"] / out["outcomes"]["median_theory"] - 1) < 0.001

paths = simulate_paths(S, T, mu, vol, 252, 60, np.random.default_rng(131))
t = np.linspace(0, T, 253)
every = 3
end = simulate_paths(S, T, mu, vol, 1, 20_000, np.random.default_rng(132))[:, -1]
bins = np.arange(50, 190.1, 5.0)
counts, _ = np.histogram(end, bins=bins)
out["fan"] = {"t": t[::every].round(4).tolist(), "paths": paths[:, ::every].round(2).tolist(),
              "mean": (S * np.exp(mu * t[::every])).round(3).tolist(),
              "median": (S * np.exp((mu - vol ** 2 / 2) * t[::every])).round(3).tolist(),
              "hist": {"y": ((bins[:-1] + bins[1:]) / 2).tolist(), "share": (counts / len(end)).round(5).tolist()},
              "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
plt.plot(t, paths.T, lw=0.5, alpha=0.5); plt.plot(t, S * np.exp(mu * t), "k", t, S * np.exp((mu - vol ** 2 / 2) * t), "k--"); plt.show()
out["outcomes"]

## 7. It's in d1 and d2

The same $-\sigma^2/2$ sits inside Black-Scholes. In $d_2 = \big[\ln(S/K) + (r - \sigma^2/2)T\big]/(\sigma\sqrt T)$ the share grows at $r - \sigma^2/2$: the risk-neutral *median* growth rate. $N(d_2)$ asks how likely the share is to finish above the strike; its median path grows at only 1.6% a year for our trade.

In [ ]:
out["d2"] = {"r_pct": r * 100, "drag_pct": vol ** 2 / 2 * 100, "median_growth_pct": (r - vol ** 2 / 2) * 100}
out["d2"]

## 8. This episode's question

A 40% volatility share with an 8% expected return: its median grows at $8\% - 0.40^2/2 = 8\% - 8\% = 0\%$. The typical outcome goes nowhere, even though the average grows at 8%.

In [ ]:
out["question"] = {"vol_pct": 40, "mu_pct": 8, "median_pct": round((0.08 - 0.40 ** 2 / 2) * 100, 10) + 0.0}
out["question"]

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")